# Phase 3 - Unified Model Optimization

This notebook consolidates three distinct hyperparameter optimization strategies explored during development:

1. **Baseline Grid Search** : Manual exploration of key architectural parameters
2. **Bayesian Optimization with Optuna** : Automated search using Optuna's TPE sampler
3. **Extended Search Space** : Comprehensive parameter exploration including layer depth and activation functions

Each section preserves the original experimental intent while using shared utility functions from the `src` module for consistency.

## Environment Setup

Load required libraries and configure the project path for consistent imports.

In [1]:
# Standard imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import optuna
from optuna.trial import TrialState
from sklearn.model_selection import train_test_split

# Project imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from data_preprocessing import resolve_project_root
from evaluate import generate_anomaly_metrics_and_threshold
from model import build_autoencoder, Autoencoder
from train import set_global_seed, train_autoencoder

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Data Loading

Load the preprocessed training and test datasets for optimization experiments.

In [2]:
# Initialize project paths
project_root = resolve_project_root()
data_dir = project_root / "data" / "processed"

# Load datasets
X_train = np.load(data_dir / "X_train.npy")
X_test = np.load(data_dir / "X_test.npy")
y_train = np.load(data_dir / "y_train.npy")
y_test = np.load(data_dir / "y_test.npy")

# Display data summary
print("Data Summary:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nFeature dimension: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Data Summary:
X_train shape: (61482, 118)
X_test shape: (84104, 118)
y_train shape: (61482,)
y_test shape: (84104,)

Feature dimension: 118
Training samples: 61482
Test samples: 84104


## Reproducibility Setup

Set random seeds for reproducible results across all optimization experiments.

In [3]:
# Set global seeds for reproducibility
set_global_seed(42)

print("Random seeds set for NumPy, TensorFlow, and Python's random module.")

Random seeds set for NumPy, TensorFlow, and Python's random module.


## Section 1: Baseline Grid Search (v1)

This section implements the original manual grid search approach, exploring a predefined set of hyperparameters to establish baseline performance bounds.

**Search Strategy:**
- Latent dimension: [8, 16, 32]
- Learning rate: [1e-3, 5e-4, 1e-4]
- Batch size: [128, 256, 512]

**Approach:** Train and evaluate each combination manually, tracking the best validation loss.

In [4]:
# Define search space for baseline grid search
latent_dims = [8, 16, 32]
learning_rates = [1e-3, 5e-4, 1e-4]
batch_sizes = [128, 256, 512]

# Storage for results
v1_results = []
best_val_loss = float('inf')
best_params = None
best_model = None

# Grid search execution
total_combinations = len(latent_dims) * len(learning_rates) * len(batch_sizes)
print(f"Testing {total_combinations} hyperparameter combinations...")

combo_count = 0
for latent_dim in latent_dims:
    for lr in learning_rates:
        for batch_size in batch_sizes:
            combo_count += 1
            print(f"\nCombination {combo_count}/{total_combinations}:")
            print(f"  Latent dim: {latent_dim}, LR: {lr}, Batch size: {batch_size}")

            # Split training data for validation
            x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

            # Build model
            model = build_autoencoder(
                input_dim=X_train.shape[1],
                latent_dim=latent_dim,
                hidden_units=32,
                dropout_rate=0.0,
                n_hidden_layers=1,
                activation_encoder='relu',
                activation_decoder='relu'
            )

            # Compile model
            optimizer = keras.optimizers.Adam(learning_rate=lr)
            model.compile(optimizer=optimizer, loss='mse')

            # Train model
            history = model.fit(
                x_fit, x_fit,
                validation_data=(x_val, x_val),
                epochs=20,
                batch_size=batch_size,
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
                verbose=0
            )

            # Record results
            val_loss = min(history.history['val_loss'])
            v1_results.append({
                'latent_dim': latent_dim,
                'learning_rate': lr,
                'batch_size': batch_size,
                'val_loss': val_loss,
                'epochs': len(history.history['loss'])
            })

            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = {'latent_dim': latent_dim, 'learning_rate': lr, 'batch_size': batch_size}
                best_model = model
                print(f"  -> New best val_loss: {val_loss:.6f}")
            else:
                print(f"  -> Val loss: {val_loss:.6f}")

print("\n" + "="*50)
print("BASELINE GRID SEARCH COMPLETE")
print("="*50)
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best parameters: {best_params}")

# Convert results to DataFrame for analysis
v1_df = pd.DataFrame(v1_results)
print("\nTop 5 configurations:")
print(v1_df.nsmallest(5, 'val_loss')[['latent_dim', 'learning_rate', 'batch_size', 'val_loss']].to_string(index=False))

Testing 27 hyperparameter combinations...

Combination 1/27:
  Latent dim: 8, LR: 0.001, Batch size: 128
  -> New best val_loss: 0.054233

Combination 2/27:
  Latent dim: 8, LR: 0.001, Batch size: 256
  -> New best val_loss: 0.053436

Combination 3/27:
  Latent dim: 8, LR: 0.001, Batch size: 512
  -> Val loss: 0.059053

Combination 4/27:
  Latent dim: 8, LR: 0.0005, Batch size: 128
  -> Val loss: 0.063214

Combination 5/27:
  Latent dim: 8, LR: 0.0005, Batch size: 256
  -> Val loss: 0.066585

Combination 6/27:
  Latent dim: 8, LR: 0.0005, Batch size: 512
  -> Val loss: 0.087816

Combination 7/27:
  Latent dim: 8, LR: 0.0001, Batch size: 128
  -> Val loss: 0.131291

Combination 8/27:
  Latent dim: 8, LR: 0.0001, Batch size: 256
  -> Val loss: 0.156141

Combination 9/27:
  Latent dim: 8, LR: 0.0001, Batch size: 512
  -> Val loss: 0.191348

Combination 10/27:
  Latent dim: 16, LR: 0.001, Batch size: 128
  -> New best val_loss: 0.019947

Combination 11/27:
  Latent dim: 16, LR: 0.001, Batc

## Section 2: Bayesian Optimization with Optuna (v2)

This section implements automated hyperparameter optimization using Optuna's Tree-structured Parzen Estimator (TPE) sampler.

**Search Strategy:**
- Latent dimension: [8, 16, 32, 64]
- Learning rate: log-uniform range [1e-4, 1e-2]
- Batch size: [32, 64, 128, 256, 512]
- Dropout rate: [0.0, 0.1, 0.2, 0.3]
- Number of hidden layers: [1, 2, 3]

**Approach:**
Optuna intelligently explores the hyperparameter space, balancing exploration and exploitation to find optimal configurations efficiently.

In [5]:
# Prepare validation split for Optuna objective
x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

# Define Optuna objective function
def objective(trial):
    """Objective function for Optuna optimization."""
    # Sample hyperparameters
    latent_dim = trial.suggest_categorical('latent_dim', [8, 16, 32, 64])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])
    dropout_rate = trial.suggest_categorical('dropout_rate', [0.0, 0.1, 0.2, 0.3])
    n_hidden_layers = trial.suggest_categorical('n_hidden_layers', [1, 2, 3])
    hidden_units = trial.suggest_categorical('hidden_units', [16, 32, 64, 128])

    # Build model with sampled hyperparameters
    model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=latent_dim,
        hidden_units=hidden_units,
        dropout_rate=dropout_rate,
        n_hidden_layers=n_hidden_layers,
        activation_encoder='relu',
        activation_decoder='relu'
    )

    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse')

    # Train model with early stopping
    history = model.fit(
        x_fit, x_fit,
        validation_data=(x_val, x_val),
        epochs=30,
        batch_size=batch_size,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        ],
        verbose=0
    )

    # Return validation loss for minimization
    return min(history.history['val_loss'])

# Create and run Optuna study
print("Starting Optuna Bayesian optimization...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=25, timeout=300)  # 25 trials or 5 minutes max

# Display results
print("\n" + "="*50)
print("OPTUNA BAYESIAN OPTIMIZATION COMPLETE")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

print("\nBest trial:")
trial = study.best_trial
print(f"  Value (validation loss): {trial.value:.6f}")
print("  Parameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Optional: Visualize optimization history
try:
    fig = optuna.visualization.plot_optimization_history(study)
    fig.show()
except Exception as e:
    print(f"Could not generate optimization history plot: {e}")

try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except Exception as e:
    print(f"Could not generate parameter importance plot: {e}")

[I 2026-07-29 19:01:13,132] A new study created in memory with name: no-name-baf4f464-656b-406b-bd2b-b83539c103d8


Starting Optuna Bayesian optimization...


/var/folders/cp/98185ksd5djf61ly4251cnw40000gn/T/ipykernel_91199/1916571214.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
[I 2026-07-29 19:01:31,262] Trial 0 finished with value: 0.023951668292284012 and parameters: {'latent_dim': 16, 'learning_rate': 0.0002051338263087451, 'batch_size': 128, 'dropout_rate': 0.1, 'n_hidden_layers': 3, 'hidden_units': 128}. Best is trial 0 with value: 0.023951668292284012.
[I 2026-07-29 19:01:33,943] Trial 1 finished with value: 0.14484471082687378 and parameters: {'latent_dim': 64, 'learning_rate': 0.0037183641805732083, 'batch_size': 512, 'dropout_rate': 0.3, 'n_hidden_layers': 1, 'hidden_units': 16}. Best is trial 0 with value: 0.023951668292284012.
[I 2026-07-29 19:01:42,001] Trial 2 finished with value: 0.0103291887


OPTUNA BAYESIAN OPTIMIZATION COMPLETE
Number of finished trials: 25

Best trial:
  Value (validation loss): 0.010329
  Parameters:
    latent_dim: 16
    learning_rate: 0.00042016720543725303
    batch_size: 256
    dropout_rate: 0.0
    n_hidden_layers: 2
    hidden_units: 128
Could not generate optimization history plot: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.
Could not generate parameter importance plot: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.


## Section 3: Extended Search Space Exploration (v3)

This section explores an expanded hyperparameter space including architectural variations and different activation functions to discover potentially superior model configurations.

**Extended Search Space:**
- Latent dimension: [8, 16, 32, 64, 128]
- Learning rate: [1e-2, 5e-3, 1e-3, 5e-4, 1e-4]
- Batch size: [32, 64, 128, 256, 512, 1024]
- Dropout rate: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
- Hidden layers: [1, 2, 3, 4]
- Hidden units: [16, 32, 64, 128, 256]
- Encoder activation: ['relu', 'tanh', 'sigmoid']
- Decoder activation: ['relu', 'sigmoid', 'linear', 'tanh']

**Approach:**
Systematic exploration of the extended space using random sampling to identify promising regions for further investigation.

In [6]:
# Define extended search space
extended_search_space = {
    'latent_dim': [8, 16, 32, 64, 128],
    'learning_rate': [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    'batch_size': [32, 64, 128, 256, 512, 1024],
    'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    'n_hidden_layers': [1, 2, 3, 4],
    'hidden_units': [16, 32, 64, 128, 256],
    'activation_encoder': ['relu', 'tanh', 'sigmoid'],
    'activation_decoder': ['relu', 'sigmoid', 'linear', 'tanh']
}

# Display search space dimensions
total_combinations = (
    len(extended_search_space['latent_dim']) *
    len(extended_search_space['learning_rate']) *
    len(extended_search_space['batch_size']) *
    len(extended_search_space['dropout_rate']) *
    len(extended_search_space['n_hidden_layers']) *
    len(extended_search_space['hidden_units']) *
    len(extended_search_space['activation_encoder']) *
    len(extended_search_space['activation_decoder'])
)

print("Extended Search Space Overview:")
for param, values in extended_search_space.items():
    print(f"  {param}: {len(values)} options")
print(f"\nTotal possible combinations: {total_combinations:,}")
print("(Too large for exhaustive search - using random sampling)")

# Random sampling approach
np.random.seed(42)
n_samples = 30  # Number of random configurations to test

v3_results = []
best_val_loss = float('inf')
best_params = None
best_model = None

print(f"\nTesting {n_samples} random configurations from extended search space...")

for i in range(n_samples):
    # Sample random hyperparameters
    latent_dim = np.random.choice(extended_search_space['latent_dim'])
    learning_rate = np.random.choice(extended_search_space['learning_rate'])
    batch_size = np.random.choice(extended_search_space['batch_size'])
    dropout_rate = np.random.choice(extended_search_space['dropout_rate'])
    n_hidden_layers = np.random.choice(extended_search_space['n_hidden_layers'])
    hidden_units = np.random.choice(extended_search_space['hidden_units'])
    activation_encoder = np.random.choice(extended_search_space['activation_encoder'])
    activation_decoder = np.random.choice(extended_search_space['activation_decoder'])

    print(f"\nConfiguration {i+1}/{n_samples}:")
    print(f"  Latent dim: {latent_dim}, LR: {learning_rate}, Batch size: {batch_size}")
    print(f"  Dropout: {dropout_rate}, Hidden layers: {n_hidden_layers}, Hidden units: {hidden_units}")
    print(f"  Encoder activation: {activation_encoder}, Decoder activation: {activation_decoder}")

    # Split training data for validation
    x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42+i, shuffle=True)

    # Build model
    model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=latent_dim,
        hidden_units=hidden_units,
        dropout_rate=dropout_rate,
        n_hidden_layers=n_hidden_layers,
        activation_encoder=activation_encoder,
        activation_decoder=activation_decoder
    )

    # Compulate model
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse')

    # Train model
    history = model.fit(
        x_fit, x_fit,
        validation_data=(x_val, x_val),
        epochs=20,
        batch_size=batch_size,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
        verbose=0
    )

    # Record results
    val_loss = min(history.history['val_loss'])
    v3_results.append({
        'latent_dim': latent_dim,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'dropout_rate': dropout_rate,
        'n_hidden_layers': n_hidden_layers,
        'hidden_units': hidden_units,
        'activation_encoder': activation_encoder,
        'activation_decoder': activation_decoder,
        'val_loss': val_loss,
        'epochs': len(history.history['loss'])
    })

    # Track best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = {
            'latent_dim': latent_dim,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'dropout_rate': dropout_rate,
            'n_hidden_layers': n_hidden_layers,
            'hidden_units': hidden_units,
            'activation_encoder': activation_encoder,
            'activation_decoder': activation_decoder
        }
        best_model = model
        print(f"  -> New best val_loss: {val_loss:.6f}")
    else:
        print(f"  -> Val loss: {val_loss:.6f}")

print("\n" + "="*50)
print("EXTENDED SEARCH SPACE COMPLETE")
print("="*50)
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best parameters: {best_params}")

# Convert results to DataFrame
v3_df = pd.DataFrame(v3_results)
print("\nTop 5 configurations:")
display_cols = ['latent_dim', 'learning_rate', 'batch_size', 'dropout_rate', 'n_hidden_layers', 'hidden_units', 'activation_encoder', 'activation_decoder', 'val_loss']
print(v3_df.nsmallest(5, 'val_loss')[display_cols].to_string(index=False))

Extended Search Space Overview:
  latent_dim: 5 options
  learning_rate: 5 options
  batch_size: 6 options
  dropout_rate: 6 options
  n_hidden_layers: 4 options
  hidden_units: 5 options
  activation_encoder: 3 options
  activation_decoder: 4 options

Total possible combinations: 216,000
(Too large for exhaustive search - using random sampling)

Testing 30 random configurations from extended search space...

Configuration 1/30:
  Latent dim: 64, LR: 0.0001, Batch size: 128
  Dropout: 0.4, Hidden layers: 1, Hidden units: 32
  Encoder activation: sigmoid, Decoder activation: linear


TypeError: 'numpy.int64' object is not iterable

## Optimization Results Summary

Comparison of the best results obtained from each optimization approach.

In [ ]:
# Create comparison DataFrame
comparison_data = []

# Add baseline results if available
if 'v1_df' in locals() and len(v1_df) > 0:
    best_v1 = v1_df.nsmallest(1, 'val_loss').iloc[0]
    comparison_data.append({
        'Approach': 'Baseline Grid Search (v1)',
        'Best Val Loss': best_v1['val_loss'],
        'Latent Dim': best_v1['latent_dim'],
        'Learning Rate': best_v1['learning_rate'],
        'Batch Size': best_v1['batch_size'],
        'Epochs': best_v1['epochs']
    })

# Add Optuna results if available
if 'study' in locals() and len(study.trials) > 0:
    best_trial = study.best_trial
    comparison_data.append({
        'Approach': 'Bayesian Optimization (v2)',
        'Best Val Loss': best_trial.value,
        'Latent Dim': best_trial.params.get('latent_dim', 'N/A'),
        'Learning Rate': best_trial.params.get('learning_rate', 'N/A'),
        'Batch Size': best_trial.params.get('batch_size', 'N/A'),
        'Dropout Rate': best_trial.params.get('dropout_rate', 'N/A'),
        'Hidden Layers': best_trial.params.get('n_hidden_layers', 'N/A'),
        'Hidden Units': best_trial.params.get('hidden_units', 'N/A')
    })

# Add extended search results if available
if 'v3_df' in locals() and len(v3_df) > 0:
    best_v3 = v3_df.nsmallest(1, 'val_loss').iloc[0]
    comparison_data.append({
        'Approach': 'Extended Search (v3)',
        'Best Val Loss': best_v3['val_loss'],
        'Latent Dim': best_v3['latent_dim'],
        'Learning Rate': best_v3['learning_rate'],
        'Batch Size': best_v3['batch_size'],
        'Dropout Rate': best_v3['dropout_rate'],
        'Hidden Layers': best_v3['n_hidden_layers'],
        'Hidden Units': best_v3['hidden_units'],
        'Encoder Activation': best_v3['activation_encoder'],
        'Decoder Activation': best_v3['activation_decoder']
    })

# Display comparison
if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("Optimization Approach Comparison:")
    print("="*80)
    print(comparison_df.to_string(index=False))
else:
    print("No optimization results available for comparison.")

# Final recommendation
print("\n" + "="*50)
print("RECOMMENDATION FOR PRODUCTION MODEL")
print("="*50)
print("Based on the optimization experiments, consider using the best performing")
print("configuration from Section 2 (Bayesian Optimization) as it typically")
print("provides the best balance of performance and efficiency.")

# Save best model from v2 if available
if 'study' in locals() and len(study.trials) > 0:
    best_trial = study.best_trial
    best_model_v2 = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=best_trial.params['latent_dim'],
        hidden_units=best_trial.params['hidden_units'],
        dropout_rate=best_trial.params['dropout_rate'],
        n_hidden_layers=best_trial.params['n_hidden_layers'],
        activation_encoder=best_trial.params['activation_encoder'],
        activation_decoder=best_trial.params['activation_decoder']
    )
    
    optimizer = keras.optimizers.Adam(learning_rate=best_trial.params['learning_rate'])
    best_model_v2.compile(optimizer=optimizer, loss='mse')
    
    # Quick retrain on full training data
    print("\nRetraining best model on full training data...")
    history = best_model_v2.fit(
        X_train, X_train,
        epochs=50,
        batch_size=best_trial.params['batch_size'],
        verbose=1
    )
    
    # Save model
    models_dir = project_root / "models"
    models_dir.mkdir(exist_ok=True)
    model_path = models_dir / "best_optimized_model.keras"
    best_model_v2.save(model_path)
    print(f"Best optimized model saved to: {model_path}")
    
    # Evaluate on test set
    print("\nEvaluating best model on test set...")
    threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
        best_model_v2, X_train, X_test, y_test, percentile=95
    )
    print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
    print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")